In [ ]:
import sys

if 'google.colab' in sys.modules:
  !pip install zombie-imp
%load_ext autoreload
%autoreload 2

import numpy as np

from pathlib import Path

if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy")
    !pip install shin
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = Path.cwd().parent

sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"

from mpp_project.daily_pipeline import run_daily_pipeline
from mpp_project.core import load_exact_scores
from mpp_project.oracle_dp import GAP_MIN, GAP_MAX, GAP_OFFSET

# ==========================================
# 0. PARAMÈTRES DU MATCH DU JOUR
# ==========================================
MATCH_DU_JOUR_ID = 3
MON_GAP_1 = -40
MON_GAP_2 = 0
HAS_BOOSTER = 1
HORIZON_NUIT = 5

# ==========================================
# 0bis. MARCHÉ DES SCORES EXACTS DU MATCH DU JOUR
# ==========================================
# Les scores exacts sont saisis dans data/exact_scores.csv (colonnes
# match_id, score, cote, crowd) — un tableur, pas du code. On charge ici les
# lignes du match du jour. Pour un nouveau match : ajouter ses lignes au CSV
# (cote = Pinnacle, crowd = % Winamax). Cote vide = score non disponible (ignoré).
# La décision du jour porte sur le SCORE EXACT ; le bonus (rareté du score) profite
# aussi à Bob (gap_1) et au peloton (gap_2). Les probas sont ANCRÉES sur le 1N2 du
# CDM_2026.csv ; un warning s'affiche si les cotes de scores s'en écartent trop.
exact_score_data = load_exact_scores(DATA_DIR / "exact_scores.csv", MATCH_DU_JOUR_ID)
print(f"📋 {len(exact_score_data)} scores chargés pour le match {MATCH_DU_JOUR_ID}.")

In [ ]:
# ==========================================
# 1. EXÉCUTION DU PIPELINE ET RECOMMANDATION (SCORES EXACTS)
# ==========================================
print(f"🚀 EXÉCUTION DU PIPELINE POUR LE MATCH {MATCH_DU_JOUR_ID}...")

reco, wr, market_df, q_table_jour = run_daily_pipeline(
    csv_path=DATA_DIR / "CDM_2026.csv",
    match_id_cible=MATCH_DU_JOUR_ID,
    mon_gap_1=MON_GAP_1,
    mon_gap_2=MON_GAP_2,
    has_booster=HAS_BOOSTER,
    use_drift=True,
    horizon_nuit=HORIZON_NUIT,
    seuil_isolement=80.0,
    nb_scenarios=3,
    near_horizon=10,
    exact_score_data=exact_score_data,   # <- décision sur le SCORE EXACT (depuis le CSV)
)

print(f"✅ Terminé ! {HORIZON_NUIT} abaques 1N2 synchronisées (projection).")

print("\n" + "="*50)
print(f"🎯 RECOMMANDATION OFFICIELLE (score exact) : {reco}")
print(f"   Espérance de Victoire (WR) : {wr*100:.2f}%")
print("="*50)

In [3]:
# ==========================================
# 1bis. DÉTAIL DU MARCHÉ DES SCORES EXACTS
# ==========================================
# market_df : un score par ligne, avec proba vraie (ancrée sur le 1N2 du CSV),
# crowd conditionnel (normalisé par outcome), bonus de points, espérance de points
# au barème simple 1/2/3, et les 3 win rates (base / keep / x2).
# Colonne pertinente selon le booster : 'WR keep (%)' si on garde le x2,
# 'WR x2 (%)' si on l'utilise, 'WR base (%)' si plus de booster.

# Meilleur WR atteignable par score (max entre garder et utiliser le booster)
if HAS_BOOSTER:
    market_df["WR best (%)"] = market_df[["WR keep (%)", "WR x2 (%)"]].max(axis=1)
else:
    market_df["WR best (%)"] = market_df["WR base (%)"]

view = market_df.sort_values("WR best (%)", ascending=False).reset_index(drop=True)

print("\n" + "=" * 78)
print("📊 MARCHÉ DES SCORES EXACTS (trié par win rate décroissant)")
print("=" * 78)
display(
    view.style
    .format({c: "{:.2f}" for c in view.columns if c.endswith("(%)") or c.startswith("E[")})
    .format({"Bonus": "{:.0f}"})
    .background_gradient(subset=["WR best (%)"], cmap="Greens")
)

# Agrégation 1N2 (contrôle : la somme des probas par outcome = P(outcome) du CSV)
agg = market_df.groupby("Outcome")["True Proba (%)"].sum().round(2)
print("\n=== Contrôle 1N2 (somme des probas de scores par outcome) ===")
print(agg.to_string())

# Top 3 espérance de points au barème simple (1 = bon outcome, 2 = + bonne diff, 3 = exact)
print("\n=== 🏅 TOP 3 ESPÉRANCE DE POINTS (barème 1/2/3) ===")
top3 = market_df.sort_values("E[pts 1/2/3]", ascending=False).head(3)
for rk, (_, row) in enumerate(top3.iterrows(), 1):
    print(f"  {rk}. {row['Score']:<5} ({row['Outcome']}) | E[pts] = {row['E[pts 1/2/3]']:.3f}")

print(f"\n🎯 Score recommandé (win rate) : {reco}  |  WR : {wr*100:.2f}%")


📊 MARCHÉ DES SCORES EXACTS (trié par win rate décroissant)


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR best (%)
0,3-0,1 (Dom),5.731355,3.039514,70,0.667837,40.178913,46.884270,43.062252,46.884270
1,3-1,1 (Dom),4.813949,4.559271,70,0.748003,40.130707,46.836143,42.960724,46.836143
2,2-0,1 (Dom),10.974595,21.276596,30,0.809610,40.118454,46.824699,42.944462,46.824699
3,0-0,N (Nul),10.100309,20.000000,50,0.645946,40.112175,46.805759,42.969460,46.805759
4,1-0,1 (Dom),14.408205,30.395137,20,0.935599,40.087593,46.793957,42.877831,46.793957
5,4-1,1 (Dom),1.892941,0.303951,100,0.629453,40.023170,46.728333,42.723118,46.728333
6,2-1,1 (Dom),9.254350,37.993921,20,0.884061,40.012955,46.719082,42.717041,46.719082
7,4-0,1 (Dom),2.289180,0.759878,70,0.585554,39.999382,46.704880,42.682110,46.704880
8,3-2,1 (Dom),2.061085,1.519757,70,0.812128,39.987389,46.692902,42.656860,46.692902
9,4-2,1 (Dom),0.769735,0.151976,100,0.707561,39.938011,46.643448,42.548653,46.643448



=== Contrôle 1N2 (somme des probas de scores par outcome) ===
Outcome
1 (Dom)    53.43
2 (Ext)    19.32
N (Nul)    27.25

=== 🏅 TOP 3 ESPÉRANCE DE POINTS (barème 1/2/3) ===
  1. 1-0   (1 (Dom)) | E[pts] = 0.936
  2. 2-1   (1 (Dom)) | E[pts] = 0.884
  3. 3-2   (1 (Dom)) | E[pts] = 0.812

🎯 Score recommandé (win rate) : 3-0 (Safe)  |  WR : 46.88%


In [4]:
# ==========================================
# 2. PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS")
print("="*50)
print("Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-60 pts/match)", "delta": -60},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+60 pts/match)", "delta": 60}
]

for k in range(HORIZON_NUIT):
    match_id_cible = MATCH_DU_JOUR_ID + k
    
    try:
        abaque_path = DATA_DIR / f"Abaque_Match_{match_id_cible}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        print(f"⚠️ Abaque introuvable pour le match {match_id_cible}. Fin de la projection.")
        break
        
    print(f"▶️ MATCH {match_id_cible} (Δt = +{k}) :")
    
    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            best_keep_idx, best_use_idx = np.argmax(wr_keep), np.argmax(wr_use)
            val_keep, val_use = wr_keep[best_keep_idx], wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)


🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?

▶️ MATCH 3 (Δt = +0) :
  🔴 Retard (-60 pts/match)    | Gaps proj. :  -40,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 46.91% | WR(x2): 42.76%
  ⚪ Maintien (Inchangé)       | Gaps proj. :  -40,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 46.91% | WR(x2): 42.76%
  🟢 Avance (+60 pts/match)    | Gaps proj. :  -40,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 46.91% | WR(x2): 42.76%
------------------------------------------------------------------------------------------
▶️ MATCH 4 (Δt = +1) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -100,  -60 | Reco : 1 (Dom) (Safe) | WR(Safe): 43.88% | WR(x2): 39.81%
  ⚪ Maintien (Inchangé)       | Gaps proj. :  -40,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 47.92% | WR(x2): 44.00%
  🟢 Avance (+60 pts/match)    | Gaps proj. :   20,   60 | Reco : 1 (Dom) (Safe) | WR(Safe): 52.32% | WR(x2): 48.50%
-------------------------------